# Advanced tutorial: build and render a templated ReactionModel

This notebook builds the templated DSL for a four-compartment monospecific anti-ligand model, renders it with `TemplatedReactionModel`, and verifies the result against the handwritten reference `model.txt`.

Comparison is order- and whitespace-insensitive, because the renderer may group components differently than the handwritten reference.

## What this tutorial adds

The basic text-model tutorial uses a small handwritten ReactionModel. Here the model has enough repeated structure to motivate templating. We will:

1. Store model dimensions as Python data.
2. Build a compact templated DSL containing `% template_globals`, `{...}` placeholder substitutions, `@`-label pairing, and `$sum` reduction functions.
3. Render the templated DSL to a concrete ReactionModel using `TemplatedReactionModel`.
4. Verify the rendered DSL against `model.txt` by normalized, order-insensitive line-count comparison.

## Setup

Imports, output file paths, and `dsl_lines` — a helper that strips comments and blank lines from DSL text. `dsl_lines` is used after each model section to report the cumulative template line count.

In [1]:
from collections import Counter
from pathlib import Path

from utilities import TemplatedReactionModel

TEMPLATED_OUTPUT_PATH = Path("advanced_templated.model")
RENDERED_OUTPUT_PATH = Path("advanced_rendered.model")
REFERENCE_PATH = Path("inputs/Templated_Model/model.txt")


def dsl_lines(text: str) -> list[str]:
    """Return non-comment, non-blank DSL lines."""
    return [line.strip() for line in text.splitlines() if line.strip() and not line.strip().startswith("#")]

## Model dimensions

Template globals define the lists that the renderer iterates over when expanding template lines. `comp` (short compartment suffix) and `compartment` (full name) are paired using the `@1` label syntax when both are needed within the same DSL line. `target` lists the three peripheral compartments connected to central by transport. `drug_arm` enumerates the two possible occupancy states of each antibody arm.

In [2]:
COMPARTMENTS = [
    ("c", "central", "2.5", "3000000000.0"),
    ("p", "peripheral", "12.8", "13000000000.0"),
    ("d", "disease", "0.1", "100000000.0"),
    ("t", "tox", "0.1", "100000000.0"),
]

TRANSPORT_LINKS = [("cp", "p"), ("cd", "d"), ("ct", "t")]

TEMPLATE_GLOBALS = {
    "comp": ["c", "p", "d", "t"],
    "compartment": ["central", "peripheral", "disease", "tox"],
    "target": ["p", "d", "t"],
    "drug_arm": ["0", "L1"],
}

TEMPLATE_GLOBALS

{'comp': ['c', 'p', 'd', 't'],
 'compartment': ['central', 'peripheral', 'disease', 'tox'],
 'target': ['p', 'd', 't'],
 'drug_arm': ['0', 'L1']}

## A tiny string builder

The builder only appends strings. Many of those strings contain template syntax — `{comp}`, `{comp@1}`, `{[S1, L1]@1}`, `$sum{..., [1, 2]}` — which is inserted verbatim. The renderer expands all placeholders during the render step.

In [3]:
template_lines: list[str] = []


def add(*lines: str) -> None:
    template_lines.extend(lines)


def blank() -> None:
    template_lines.append("")


def comment(text: str = "") -> None:
    template_lines.append(f"# {text}" if text else "#")


def add_template_header() -> None:
    add(
        "%% ReactionModel@2",
        "",
        "initialization = initial_value()",
        "time_unit = s",
        "default_state_unit = nmol",
        "",
        "% template_globals",
    )
    for name, values in TEMPLATE_GLOBALS.items():
        add(f"{name} = [{', '.join(values)}]")
    add("", "% components")


add_template_header()

## Parameters

Compartment volumes differ per compartment and are written as explicit lines. Parameters that share the same value across all compartments use template lines instead — for example, `css_S1_{comp}_nM :nM := 0` expands to one parameter per compartment.

In [4]:
comment("Parameters")

for suffix, _name, volume, _cells in COMPARTMENTS:
    add(f"v_{suffix}_L :L := {volume}")

add(
    "css_R1_total_{comp}_rpc :1 := 10000",
    "css_S1_{comp}_nM :nM := 0",
    "css_L1_free_{comp}_nM :nM := 0.05",
    "thalf_R1_hr :hr := 1",
    "thalf_S1_hr :hr := 0.5",
    "thalf_L1_hr :hr := 0.5",
    "thalf_ab_day :d := 28",
    "scale_thalf_ab_{comp} :1 := 1",
    "tdist_c{target}_{[S1, L1]@1}_hr :hr := 30",
    "pdist_cp_ab :1 := 0.190625",
    "pdist_cd_ab :1 := 0.3",
    "pdist_ct_ab :1 := 0.3",
    "tdist_c{target}_ab_hr :hr := 30",
)

cell_number_values = [cell_number for _suffix, _name, _volume, cell_number in COMPARTMENTS]
add(f"R_cell_number_{{comp@1}} :1 := {{[{', '.join(cell_number_values)}]@1}}")

add(
    "R_cell_diameter_um :um = 10",
    "kon_ab_L1 :(1/nM/s) := 1e-3",
    "Kd_ab_L1_nM :nM := 0.1",
    "valency_ab_L1 :1 := 2",
    "scale_Kd_ab_L1_{comp} :1 := 1",
    "kon_L1_R1 :(1/nM/s) := 1e-3",
    "Kd_L1_R1_nM :nM := 1",
    "MW_gmol :(g/mol) := 150000",
    "dose_n :1 := 7",
    "dose_amount_mg :mg := 100",
    "dose_interval_day :d := 14",
    "BW_kg :kg := 70",
    "F :1 := 1",
    "thalf_abs_day :d := 2.5",
)

len(dsl_lines("\n".join(template_lines)))

42

## Derived quantities

Derived parameters illustrate the conciseness gained from templating: the algebraic expressions remain legible because only the repeated identifiers are parametrized. The inline lists `{[L1_free, S1]@1}` and `{[L1, S1]@1}` share label `@1`, so the renderer iterates them in lockstep, producing one expanded line per pair.

In [5]:
blank()
comment("Derived parameters")

add(
    "Kd_ab_L1_{comp}_nM = Kd_ab_L1_nM * scale_Kd_ab_L1_{comp}",
    "thalf_ab_{comp}_day = thalf_ab_day * scale_thalf_ab_{comp}",
    "sec_per_hr = 3600:(s/hr)",
    "sec_per_day = sec_per_hr * 24: (hr/d)",
    "thalf_{[abs, L1, R1, S1]@1} = thalf_{[abs_day, L1_hr, R1_hr, S1_hr]@1} * sec_per_{[day, hr, hr, hr]@1}",
    "tdist_c{target}_{[L1, S1]@1} = tdist_c{target}_{[L1, S1]@1}_hr * sec_per_hr",
    "thalf_ab_{comp} = thalf_ab_{comp}_day * sec_per_day",
    "tdist_c{target}_ab = tdist_c{target}_ab_hr * sec_per_hr",
    "number_per_nmol = 6.022e14 :(1/nmol)",
    "ksyn_L1_{comp} = css_L1_free_{comp}_nM*log(2)*(Kd_L1_R1_nM*kon_L1_R1*number_per_nmol*thalf_R1*v_{comp}_L + R_cell_number_{comp}*css_R1_total_{comp}_rpc*kon_L1_R1*thalf_L1 + css_L1_free_{comp}_nM*kon_L1_R1*number_per_nmol*thalf_R1*v_{comp}_L + log(2)*number_per_nmol*v_{comp}_L)/(number_per_nmol*thalf_L1*(Kd_L1_R1_nM*kon_L1_R1*thalf_R1 + css_L1_free_{comp}_nM*kon_L1_R1*thalf_R1 + log(2)))",
    "ksyn_R1_{comp} = R_cell_number_{comp}*css_R1_total_{comp}_rpc*log(2)/(number_per_nmol*thalf_R1)",
    "ksyn_S1_{comp} = css_S1_{comp}_nM*log(2)*v_{comp}_L/thalf_S1",
    "kc{target}_{[L1, S1]@1}  =  css_{[L1_free, S1]@1}_{target}_nM*v_{target}_L*log(2)/(tdist_c{target}_{[L1, S1]@1}*(css_{[L1_free, S1]@1}_c_nM*v_c_L + css_{[L1_free, S1]@1}_{target}_nM*v_{target}_L + 1e-16:nmol))",
    "k{target}c_{[L1, S1]@1}  =  css_{[L1_free, S1]@1}_c_nM*v_c_L*log(2)/(tdist_c{target}_{[L1, S1]@1}*(css_{[L1_free, S1]@1}_c_nM*v_c_L + css_{[L1_free, S1]@1}_{target}_nM*v_{target}_L + 1e-16:nmol))",
)

len(dsl_lines("\n".join(template_lines)))

56

## Compartments and states

The line `{compartment@1} ~ 3 = v_{comp@1}_L` pairs each full compartment name with its short suffix via the shared `@1` label. The antibody state line uses three `@` labels: `@1` and `@2` jointly enumerate the four arm-occupancy combinations (`0_0`, `0_L1`, `L1_0`, `L1_L1`), and `@3` associates each resulting state with its compartment.

In [6]:
blank()
comment("Compartments and states")

add(
    "{compartment@1} ~ 3 = v_{comp@1}_L",
    "um_per_dm = 1e5 :(um/dm)",
    "pi :1 = 3.141592653589793",
    "R_cell_diameter = R_cell_diameter_um / um_per_dm",
    "membrane_{comp} ~ 2 = R_cell_number_{comp} * pi * R_cell_diameter^2 + 1e-16:dm^2",
    "L1_R1_{comp}_0 = R_cell_number_{comp}*css_L1_free_{comp}_nM*css_R1_total_{comp}_rpc*kon_L1_R1*thalf_R1/(number_per_nmol*(Kd_L1_R1_nM*kon_L1_R1*thalf_R1 + css_L1_free_{comp}_nM*kon_L1_R1*thalf_R1 + log(2)))",
    "L1_R1_{comp} @ membrane_{comp} *= L1_R1_{comp}_0",
    "free_L1_{comp}_0 = css_L1_free_{comp}_nM*v_{comp}_L",
    "L1_{comp@1} @ {compartment@1} *= free_L1_{comp@1}_0",
    "free_R1_{comp}_0 = R_cell_number_{comp}*css_R1_total_{comp}_rpc*(Kd_L1_R1_nM*kon_L1_R1*thalf_R1 + log(2))/(number_per_nmol*(Kd_L1_R1_nM*kon_L1_R1*thalf_R1 + css_L1_free_{comp}_nM*kon_L1_R1*thalf_R1 + log(2)))",
    "R1_{comp} @ membrane_{comp} *= free_R1_{comp}_0",
    "total_R1_{comp}_0 = free_R1_{comp}_0 + L1_R1_{comp}_0",
    "total_L1_{comp}_0 = free_L1_{comp}_0 + L1_R1_{comp}_0",
    "free_S1_{comp}_0 = css_S1_{comp}_nM * v_{comp}_L",
    "total_S1_{comp}_0 = free_S1_{comp}_0",
    "S1_{comp@1} @ {compartment@1} *= free_S1_{comp@1}_0",
    "ab_{drug_arm@1}_{drug_arm@2}_{comp@3} @ {compartment@3} *= 0",
    "ab_0_0_sc @ central *= 0",
)

len(dsl_lines("\n".join(template_lines)))

74

## Routes and reactions

Routes are written with empty schedules, rendered as `@()`. The reference file uses `null` to represent an unscheduled route — both are equivalent. Reactions use the same template globals to express zero-order synthesis, first-order degradation, reversible binding, and inter-compartment transport.

In [7]:
blank()
comment("Routes and reactions")

add(
    "IV_mg :mg = @(); ab_0_0_c  += amt * 1e6:(ng/mg) / MW_gmol",
    "SC_mg :mg = @(); ab_0_0_sc += amt * F * 1e6:(ng/mg) / MW_gmol",
    "ab_0_0_sc -> ab_0_0_c; thalf=thalf_abs",
    "-> {[R1, L1, S1]@1}_{comp}; kf=ksyn_{[R1, L1, S1]@1}_{comp}",
    "{[R1, S1, L1, L1_R1]@1}_{comp} ->; thalf=thalf_{[R1, S1, L1, R1]@1}",
    "ab_{drug_arm@1}_{drug_arm@2}_{comp@3} -> ; thalf=thalf_ab_{comp@3}",
    "L1_{comp} + R1_{comp} <-> L1_R1_{comp}; kd=Kd_L1_R1_nM, kon=kon_L1_R1",
    "ab_{[0, L1]@1}_{[0, 0]@1}_{comp} + L1_{comp} <-> ab_{[0, L1]@1}_{[L1, L1]@1}_{comp}; kd=Kd_ab_L1_{comp}_nM, kon=kon_ab_L1 * (valency_ab_L1 - 1)",
    "ab_{[0, 0]@1}_{[0, L1]@1}_{comp} + L1_{comp} <-> ab_{[L1, L1]@1}_{[0, L1]@1}_{comp}; kd=Kd_ab_L1_{comp}_nM, kon=kon_ab_L1",
    "S1_c <-> S1_{target}; kf=kc{target}_S1, kr=k{target}c_S1",
    "L1_c <-> L1_{target}; kf=kc{target}_L1, kr=k{target}c_L1",
    "ab_{drug_arm@1}_{drug_arm@2}_c <-> ab_{drug_arm@1}_{drug_arm@2}_{target@3}; pdist=pdist_c{target@3}_ab, tdist=tdist_c{target@3}_ab",
)

len(dsl_lines("\n".join(template_lines)))

86

## Outputs

Output expressions are assignments and follow the same template syntax as parameters and derived quantities. `$sum{ab_{drug_arm@1}_{drug_arm@2}_{comp}, [1,2]}` sums over both `@1` and `@2` label dimensions, collapsing all four arm-occupancy species into a single total for each compartment.

In [8]:
blank()
comment("Outputs")

add(
    "total_L1_{comp}_nmol :nmol = L1_{comp} + L1_R1_{comp} + 1*ab_0_L1_{comp} + 1*ab_L1_0_{comp} + 2*ab_L1_L1_{comp}",
    "total_L1_{comp}_nM :nM = total_L1_{comp}_nmol / v_{comp}_L",
    "free_L1_{comp}_nmol :nmol = L1_{comp}",
    "free_L1_{comp}_nM :nM = L1_{comp} / v_{comp}_L",
    "free_R1_{comp}_nmol :nmol = R1_{comp}",
    "free_R1_{comp}_nM :nM = R1_{comp} / v_{comp}_L",
    "total_R1_{comp}_nmol :nmol = R1_{comp} + L1_R1_{comp}",
    "total_R1_{comp}_nM :nM = total_R1_{comp}_nmol / v_{comp}_L",
    "total_S1_{comp}_nmol :nmol = S1_{comp}",
    "total_S1_{comp}_nM :nM = total_S1_{comp}_nmol / v_{comp}_L",
    "free_S1_{comp}_nmol :nmol = S1_{comp}",
    "free_S1_{comp}_nM :nM = free_S1_{comp}_nmol / v_{comp}_L",
    "L1_bound_to_drug_{comp}_nmol :nmol = total_L1_{comp}_nmol - L1_{comp} - L1_R1_{comp}",
    "drug_bound_to_L1_{comp}_nmol :nmol = ab_0_L1_{comp} + ab_L1_0_{comp} + ab_L1_L1_{comp}",
    "drug_bound_to_L1_{comp}_nM :nM = drug_bound_to_L1_{comp}_nmol / v_{comp}_L",
    "total_soluble_drug_{comp}_nM :nM = ($sum{ab_{drug_arm@1}_{drug_arm@2}_{comp}, [1,2]}) / v_{comp}_L",
    "total_soluble_drug_{comp}_ngmL :(ng/mL) = total_soluble_drug_{comp}_nM * MW_gmol / 1000:(mL/L)",
    "unbound_drug_{comp}_nM :nM = ab_0_0_{comp} / v_{comp}_L",
    "unbound_drug_{comp}_ngmL :(ng/mL) = unbound_drug_{comp}_nM * MW_gmol / 1000:(mL/L)",
    "total_drug_{comp}_nmol :nmol = $sum{ab_{drug_arm@1}_{drug_arm@2}_{comp}, [1,2]}",
    "total_drug_{comp}_nM :nM = total_drug_{comp}_nmol / v_{comp}_L",
    "total_drug_{comp}_mg :mg = total_drug_{comp}_nmol * (1e3/1e9):(mmol/nmol) * MW_gmol",
    "inhibition_L1_R1_{comp}_perc :1 = 100 * (1 - L1_R1_{comp} / (L1_R1_{comp}[0:s] + 1e-16:nmol))",
    "target_engagement_L1_{comp}_perc :1 = 100 * L1_bound_to_drug_{comp}_nmol / (total_L1_{comp}_nmol + 1e-16:nmol)",
    "activation_R1_{comp}_perc :1 = 100 * (total_R1_{comp}_nmol - R1_{comp}) / (total_R1_{comp}_nmol + 1e-16:nmol)",
)

len(dsl_lines("\n".join(template_lines)))

111

## Write the templated model

At this point the file contains the compact templated representation — `{...}` placeholders are still present and have not been expanded. The assertions confirm this before the render step.

In [9]:
templated_model_text = "\n".join(template_lines).rstrip() + "\n"
TEMPLATED_OUTPUT_PATH.write_text(templated_model_text, encoding="utf-8")

assert "% template_globals" in templated_model_text
assert "{comp}" in templated_model_text
assert "$sum" in templated_model_text

print(f"Wrote {TEMPLATED_OUTPUT_PATH}")
print(f"Templated DSL lines: {len(dsl_lines(templated_model_text))}")
print("\n".join(templated_model_text.splitlines()[:35]))

Wrote advanced_templated.model
Templated DSL lines: 111
%% ReactionModel@2

initialization = initial_value()
time_unit = s
default_state_unit = nmol

% template_globals
comp = [c, p, d, t]
compartment = [central, peripheral, disease, tox]
target = [p, d, t]
drug_arm = [0, L1]

% components
# Parameters
v_c_L :L := 2.5
v_p_L :L := 12.8
v_d_L :L := 0.1
v_t_L :L := 0.1
css_R1_total_{comp}_rpc :1 := 10000
css_S1_{comp}_nM :nM := 0
css_L1_free_{comp}_nM :nM := 0.05
thalf_R1_hr :hr := 1
thalf_S1_hr :hr := 0.5
thalf_L1_hr :hr := 0.5
thalf_ab_day :d := 28
scale_thalf_ab_{comp} :1 := 1
tdist_c{target}_{[S1, L1]@1}_hr :hr := 30
pdist_cp_ab :1 := 0.190625
pdist_cd_ab :1 := 0.3
pdist_ct_ab :1 := 0.3
tdist_c{target}_ab_hr :hr := 30
R_cell_number_{comp@1} :1 := {[3000000000.0, 13000000000.0, 100000000.0, 100000000.0]@1}
R_cell_diameter_um :um = 10
kon_ab_L1 :(1/nM/s) := 1e-3
Kd_ab_L1_nM :nM := 0.1


## Render with repository tooling

`TemplatedReactionModel.from_text()` parses the templated DSL. `to_reaction_model()` expands all templates and writes a concrete ReactionModel file.

In [10]:
templated_model = TemplatedReactionModel.from_text(templated_model_text)
if not isinstance(templated_model, TemplatedReactionModel):
    raise ValueError(templated_model)

templated_model.to_reaction_model(RENDERED_OUTPUT_PATH)
rendered_model_text = RENDERED_OUTPUT_PATH.read_text(encoding="utf-8")

print(f"Wrote {RENDERED_OUTPUT_PATH}")
print(f"Rendered DSL lines: {len(dsl_lines(rendered_model_text))}")
print("\n".join(rendered_model_text.splitlines()[:35]))

Wrote advanced_rendered.model
Rendered DSL lines: 370
%% ReactionModel@2

initialization = initial_value()
time_unit = s
% components

 # Parameters
v_c_L:L := 2.5
v_p_L:L := 12.8
v_d_L:L := 0.1
v_t_L:L := 0.1
css_R1_total_c_rpc:1 := 10000
css_R1_total_p_rpc:1 := 10000
css_R1_total_d_rpc:1 := 10000
css_R1_total_t_rpc:1 := 10000
css_S1_c_nM:nM := 0
css_S1_p_nM:nM := 0
css_S1_d_nM:nM := 0
css_S1_t_nM:nM := 0
css_L1_free_c_nM:nM := 0.05
css_L1_free_p_nM:nM := 0.05
css_L1_free_d_nM:nM := 0.05
css_L1_free_t_nM:nM := 0.05
thalf_R1_hr:hr := 1
thalf_S1_hr:hr := 0.5
thalf_L1_hr:hr := 0.5
thalf_ab_day:d := 28
scale_thalf_ab_c:1 := 1
scale_thalf_ab_p:1 := 1
scale_thalf_ab_d:1 := 1
scale_thalf_ab_t:1 := 1
tdist_cp_S1_hr:hr := 30
tdist_cp_L1_hr:hr := 30
tdist_cd_S1_hr:hr := 30
tdist_cd_L1_hr:hr := 30


## Verify against `model.txt`

The rendered file is verified to have the same model structure as `model.txt`. Comparison treats lines as a multiset — duplicates count, order does not.

### Comparison helpers

`model.txt` is a handwritten `ReactionModel@1` file. The renderer produces `ReactionModel@2`, uses `initial_value()` instead of `InitialValue()`, emits explicit `:nmol` state units rather than a `default_state_unit` directive, writes empty schedules as `@()`, and deparses some numeric literals to decimal. These are syntax-level differences with no structural significance. The two functions below normalize these spellings so that the line-count comparison is meaningful.

In [ ]:
def normalize_line(line: str, *, reference: bool = False) -> str | None:
    """Normalize equivalent DSL spellings before structural comparison.

    `model.txt` is an older handwritten `ReactionModel@1` file. The renderer writes
    `ReactionModel@2`, uses `initial_value()`, emits explicit `:nmol` state units
    instead of a `default_state_unit` line, writes empty schedules as `@()`, and
    deparses some numeric literals in decimal form. These are syntax differences,
    not structural model differences.
    """
    compact = "".join(line.split())

    if reference and compact == "default_state_unit=nmol":
        return None

    replacements = {
        "ReactionModel@1": "ReactionModel@2",
        "InitialValue()": "initial_value()",
        "=null;": "=@();",
        "0.001": "1e-3",
        "602200000000000.0": "6.022e14",
        "100000.0": "1e5",
        "1000000.0": "1e6",
        "1000.0/1000000000.0": "1e3/1e9",
    }
    for old, new in replacements.items():
        compact = compact.replace(old, new)

    if reference and "*=" in compact and "@" in compact and ":nmol" not in compact:
        left, right = compact.split("*=", 1)
        compact = f"{left}:nmol*={right}"

    return compact


def normalized_counts(text: str, *, reference: bool = False) -> Counter[str]:
    normalized = []
    for line in dsl_lines(text):
        line = normalize_line(line, reference=reference)
        if line is not None:
            normalized.append(line)
    return Counter(normalized)

In [11]:
reference_text = REFERENCE_PATH.read_text(encoding="utf-8")
reference_counts = normalized_counts(reference_text, reference=True)
rendered_counts = normalized_counts(rendered_model_text)

missing = reference_counts - rendered_counts
extra = rendered_counts - reference_counts

print(f"Reference normalized DSL lines: {sum(reference_counts.values())}")
print(f"Rendered normalized DSL lines: {sum(rendered_counts.values())}")
print(f"Missing lines: {sum(missing.values())}")
print(f"Extra lines: {sum(extra.values())}")

if missing or extra:
    print("\nMissing examples:")
    for line, count in missing.most_common(10):
        print(count, line)
    print("\nExtra examples:")
    for line, count in extra.most_common(10):
        print(count, line)
    raise AssertionError("Rendered model does not structurally match model.txt")

print("Structural match confirmed.")

Reference normalized DSL lines: 370
Rendered normalized DSL lines: 370
Missing lines: 0
Extra lines: 0
Structural match confirmed.
